In [68]:
import matplotlib as plt
import sklearn
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

In [69]:
# Needed functions

def read(name):
    return pd.read_csv(f"dataset/{name}.csv", na_values=["\\N"])

def lap_milliseconds(t):
    if pd.isna(t) or ":" not in str(t):
        return np.nan
    minutes, seconds = str(t).split(":")
    return int(minutes) * 60 * 1000 + float(seconds) * 1000
    

In [70]:
# reading dataset

races = read("races")
results = read("results")
qualifying = read("qualifying")
drivers = read("drivers")
circuits = read("circuits")
constructors_standings = read("constructor_standings")
constructors = read("constructors")
sprints = read("sprint_results")

# we will only choose races from 2014 until present
# because this period is hybrid engines period and much more usable
races = races[races["raceId"] > 899]
results = results[results["raceId"].isin(races["raceId"])]
qualifying = qualifying[qualifying["raceId"].isin(races["raceId"])]
constructors_standings = constructors_standings[constructors_standings["raceId"].isin(races["raceId"])]

# keeping needed columns
races = races[["raceId", "year", "round", "circuitId"]]
results = results[["raceId","driverId","constructorId","grid", "positionOrder"]]
qualifying = qualifying[[ "raceId", "driverId", "constructorId", "q1", "q2", "q3" ]]
constructors_standings = constructors_standings[["raceId", "constructorId"]]

results = results.rename(columns={"positionOrder": "final_result", "grid": "race_grid"})
sprints = sprints.rename(columns={"positionOrder": "sprint_result", "grid": "sprint_grid"})
sprints = sprints[[ "raceId", "driverId", "constructorId", "sprint_grid", "sprint_result" ]]

In [71]:
# creating a single dataframe with all the needed information

df = (
    races
    .merge(results, on="raceId", how="left")
    .merge(
        qualifying,
        on=["raceId", "driverId", "constructorId"],
        how="left"
    )
    .merge(
        constructors_standings,
        on=["raceId", "constructorId"],
        how="left"
    )
    .merge(
        sprints,
        on=["raceId", "driverId", "constructorId"],
        how="left",
    )
)

# change laptime to milliseconds
df["q1"] = df["q1"].apply(lap_milliseconds)
df["q2"] = df["q2"].apply(lap_milliseconds)
df["q3"] = df["q3"].apply(lap_milliseconds)

# deal with nan or 0
df.dropna(subset=['driverId'], inplace = True)
df.dropna(subset=['constructorId'], inplace = True)
df.dropna(subset=['raceId'], inplace = True)

# race grid of 0 means start in the pit lane, so we will set it to the back of grid, same for nan values
df.loc[df["race_grid"] == 0, "race_grid"] = df["race_grid"].max()
df.loc[df["race_grid"].isna(), "race_grid"] = df["race_grid"].max()

df["is_sprint_weekend"] = df["sprint_result"].notna().astype(int)
df[["sprint_grid", "sprint_result"]] = df[["sprint_grid", "sprint_result"]].fillna(0)

df["q1_reached"] = df["q1"].notna().astype("int")
df["q2_reached"] = df["q2"].notna().astype("int")
df["q3_reached"] = df["q3"].notna().astype("int")

for c in ["q1", "q2", "q3"]:
    fastest = df.groupby("raceId")[c].transform("min")
    df[c] = (fastest / df[c]).fillna(0)   # pole positon is 1 and others are less than 1, if no time set then 0


df["driverId"] = df["driverId"].astype("int")
df["constructorId"] = df["constructorId"].astype("int")
df["raceId"] = df["raceId"].astype("int")
df["race_grid"] = df["race_grid"].astype("int")
df["sprint_grid"] = df["sprint_grid"].astype("int")
df["sprint_result"] = df["sprint_result"].astype("int")
df["final_result"] = df["final_result"].astype("int")
df["podium"] = (df["final_result"] <= 3).astype("int")
df.head(5)


,raceId,year,round,circuitId,driverId,constructorId,race_grid,final_result,q1,q2,q3,sprint_grid,sprint_result,is_sprint_weekend,q1_reached,q2_reached,q3_reached,podium
0,900,2014,1,1,3,131,3,1,0.980673,1.000000,0.996520,0,0,0,1,1,1,1
1,900,2014,1,1,825,1,4,2,0.998087,0.990479,0.985683,0,0,0,1,1,1,1
2,900,2014,1,1,18,1,10,3,0.993205,0.979193,0.000000,0,0,0,1,1,0,1
3,900,2014,1,1,4,6,5,4,0.993292,0.994738,0.984993,0,0,0,1,1,1,0
4,900,2014,1,1,822,3,15,5,0.990983,0.984709,0.963790,0,0,0,1,1,1,0


In [72]:
X = df[["race_grid", "q1", "q2", "q3", "sprint_grid", "sprint_result", "is_sprint_weekend", "q1_reached", "q2_reached", "q3_reached"]]
Y = df["podium"]

In [73]:
X_train, X_test, y_train, y_test = train_test_split( X, Y, test_size=0.2, random_state=4)


In [74]:
# the best recall i could get for logistic regression was 60% with this model
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix , classification_report

model = make_pipeline(StandardScaler(), LogisticRegression(C = 0.8,solver='saga' ,penalty='elasticnet', max_iter=1000))

model.fit(X_train, y_train)
yhat = model.predict(X_test)

print(confusion_matrix(y_test, yhat, labels=[1,0]))
print(classification_report(y_test, yhat))

[[ 97  66]
 [ 49 884]]
              precision    recall  f1-score   support

           0       0.93      0.95      0.94       933
           1       0.66      0.60      0.63       163

    accuracy                           0.90      1096
   macro avg       0.80      0.77      0.78      1096
weighted avg       0.89      0.90      0.89      1096



e:\machine learning\projects\predict-f1-winner\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


In [88]:
from sklearn.svm import SVC

model = make_pipeline(StandardScaler(), SVC(kernel='rbf', C=0.1, gamma='auto'))
model.fit(X_train, y_train)
yhat = model.predict(X_test)

print(confusion_matrix(y_test, yhat, labels=[1,0]))
print(classification_report(y_test, yhat))

[[105  58]
 [ 51 882]]
              precision    recall  f1-score   support

           0       0.94      0.95      0.94       933
           1       0.67      0.64      0.66       163

    accuracy                           0.90      1096
   macro avg       0.81      0.79      0.80      1096
weighted avg       0.90      0.90      0.90      1096

